# Case study ``pyaesa`` workflow for the consensus process

This notebook runs the absolute sustainability ratio (ASR) uncertainty analysis for the case study of the consensus process.\
It estimates the LCA Monte Carlo directly based on io_lca, computes the ACC and finally provides the results for ASR over the period of time considered.\
This notebook allows for generating all the figures and results of the case study, ensuring full transparency and reproducibility.\
This notebook can be used for L1 and L2 allocation levels (countries and sectors).

# `pyaesa` installation

This notebook uses the `pyaesa` Python package (v1.2.8). Install the release from PyPI before running the workflow:

```bash
python -m pip install pyaesa
```

For package documentation, API reference, and tutorials, see [pyaesa.readthedocs.io](https://pyaesa.readthedocs.io/). 

The source code is available on GitHub at [AESAtoolkit/pyaesa](https://github.com/AESAtoolkit/pyaesa).

# Imports

In [ ]:
from pyaesa import (
    set_workspace,
    download_ar6,
    download_mrio,
    download_pop_gdp,
    process_mrio,
    process_pop_gdp,
    uncertainty_asr,
    uncertainty_io_lca,
    deterministic_asocc,
    disaggregate_asocc,
)

import shutil
from datetime import datetime
from matplotlib import pyplot as plt
import glob
import os
import imageio.v3 as iio

# Parameters

In [ ]:
WORKSPACE_TOP = r"C:\path\to\your\pyaesa-workspace"  # replace this with the path of your workspace

Re-run the whole notebook for each functional unit targeted in the case study:\
``fu_code_= "L1.a"``\
``fu_code_= "L1.b"``

``fu_code_= "L2.a.b"``\
``fu_code_= "L2.a.c"``\
``fu_code_= "L2.c.a"``\
``fu_code_= "L2.c.b"``

Comment or uncomment functional unit selector below, and re-run all the notebook.\
Everything else will be handled automatically.

Alternatively, you can also run the script ``all_fus_notebook.ipynb`` in ./scripts to handle that for you.

In [ ]:
# Functional unit selector
# fu_code_ = "L1.a"
# fu_code_ = "L1.b"
fu_code_ = "L2.a.b"
# fu_code_ = "L2.a.c"
# fu_code_ = "L2.c.a"
# fu_code_ = "L2.c.b"

In [ ]:
# (EE-)MRIO selection
source_ = "exiobase_3102_ixi"

# (Dis)aggregation (country and/or sector)
agg_reg_ = False
agg_reg_name_ = None

region_sel = ["NL"]
r_c_ = region_sel
r_f_ = region_sel
r_p_ = region_sel
folder_name_region_sel = "_".join(region_sel) if len(region_sel) < 7 else len(region_sel)

agg_sec_ = True
agg_sec_name_ = "ELECTRICITY"
s_p_ = [agg_sec_name_]

if agg_reg_ and agg_sec_:
    agg_version_ = f"{agg_sec_name_}-{agg_reg_name_}"
elif agg_reg_:
    agg_version_ = f"{agg_reg_name_}"
elif agg_sec_:
    agg_version_ = f"{agg_sec_name_}"
else:
    agg_version_ = None

# Project name
project_name_ = f"{agg_version_}/{folder_name_region_sel}/FU-{fu_code_}_{agg_version_}"
print(f"Project name: {project_name_}")

# Time period
study_period_ = range(1995, 2022 + 1)  # range(1995, 2022+1) #range(1995, 2022+1)
prospective_study_period_ = range(study_period_[-1], 2040 + 1)
dynamic_study_period_ = range(2015, study_period_[-1])

# LCIA methods
lcia_method_ = ["gwp100_lcia", "pb_lcia"]  # "ef_3.1"

# asocc lcia_uncertainty parameters
sector_cov_mapping_ = {agg_sec_name_: "Electricity"}

# General Monte Carlo parameters
mc_convergence_ = True  # default=True
mc_n_runs_ = 1000
mc_max_runs_ = 100000

# General figures parameters
subfigures_ = True  # default=True

# General Sobol parameters
sobol_active_ = True  # default=False

# Run the dynamic AR6 part
run_dynamic_ar6_b = False

# Generate GIF with polars
generate_gif_b = False  # Set to 'False' to speed up the code

In [ ]:
# Set relevant functional units for the region selectors r_c, r_f, and r_p.
FU_FOR_S_P = ["L2.a.a", "L2.a.b", "L2.a.c", "L2.b.a", "L2.b.b", "L2.c.a", "L2.c.b"]
FU_FOR_R_C = ["L2.b.b", "L2.c.b"]  # L2.*.b
FU_FOR_R_F = ["L1.a", "L2.b.a", "L2.c.a"]  # L2.*.a
FU_FOR_R_P = ["L1.b", "L2.a.a", "L2.a.b", "L2.a.c", "L2.b.a", "L2.b.b"]  # L2.a.*, L2.b.*

In [ ]:
timestamp_start = datetime.now()

# Initialize the workspace

`set_workspace(...)` creates or selects the pyaesa workspace where the LCA notebook staged the external LCA sources and where the ASR outputs will be written.


In [ ]:
set_workspace(WORKSPACE_TOP)

# Download and process `pyaesa` input data


### Copy settings files for aggregation to the workspace

The files are automatically copied to the relevant location.

In [ ]:
def check_and_copy_files(filename, destination):
    files_dir = os.getcwd().split("scripts")[0] + "files\\"
    dest_path = destination + filename
    if not os.path.exists(dest_path):
        shutil.copy2(f"{files_dir}{filename}", dest_path)
        print(f"{filename} copied to {dest_path}")
    else:
        print(f"{filename} already exists in {dest_path}")

    return None

In [ ]:
check_and_copy_files(
    filename="agg_sec_ELECTRICITY.csv",
    destination=f"{WORKSPACE_TOP.replace('\\', '/')}/data_raw/mrio/exiobase_3/aggregation/ixi/",
)

check_and_copy_files(
    filename="agg_sec_oecd_d.csv",
    destination=f"{WORKSPACE_TOP.replace('\\', '/')}/data_raw/mrio/exiobase_3/aggregation/ixi/",
)

check_and_copy_files(
    filename="agg_reg_NL.csv",
    destination=f"{WORKSPACE_TOP.replace('\\', '/')}/data_raw/mrio/oecd_v2025/aggregation/",
)

### AR6 climate scenario data

Download the AR6 climate scenario data required for dynamic AR6 carrying capacities.


In [ ]:
download_ar6()

### Population and GDP data

Download and process population and GDP data needed for computing allocated shares.


In [ ]:
download_pop_gdp()

In [ ]:
process_pop_gdp()

### MRIO data

Download and process the MRIO tables needed for computing allocated shares.

The `ELECTRICITY` EXIOBASE sector aggregation groups the different EXIOBASE electricity sectors together (aggregation csv provided by `pyaesa`).

(The `oecd_d` EXIOBASE sector aggregation groups EXIOBASE electricity, gas, and water sectors to match OECD ICIO sector D, `Electricity, gas, steam and air conditioning supply` (aggregation csv provided by `pyaesa`).

The OECD `NL` region aggregation renames the OECD ICIO region code `NLD` to `NL` so it matches EXIOBASE Netherlands code in the disaggregation step.)


In [ ]:
download_mrio("exiobase_3102_ixi")

In [ ]:
download_mrio("oecd_v2025")

Process MRIO in original classification (no aggregation for regions or sectors)

In [ ]:
# process_mrio(
#     source=source_,
#     years=range(1995,2024+1) if 'exio' in source_ else range(1995,2022+1),
#     lcia_method=lcia_method_ if 'exio' in source_ else None,
#     refresh=False,
# )

Process MRIO if aggregation is required

In [ ]:
process_mrio(
    source=source_,
    years=range(1995, 2024 + 1) if "exio" in source_ else range(1995, 2022 + 1),
    lcia_method=lcia_method_ if "exio" in source_ else None,
    agg_reg=agg_reg_,
    agg_sec=agg_sec_,
    agg_version=agg_version_,
    refresh=False,
)


# Prepare disaggregated aSoCC for inter-MRIO uncertainty

The notebook computes three deterministic aSoCC prerequisite scopes: OECD ICIO sector D target, EXIOBASE grouped to OECD ICIO sector D, and EXIOBASE electricity reference scope.

`disaggregate_asocc(...)` then uses the two EXIOBASE reference scopes to split the OECD ICIO sector D aSoCC target to EXIOBASE electricity sector resolution and writes the `oecd_electricity` source used by inter-MRIO uncertainty.


#### Processing of MRIOs included in inter-MRIO uncertainty

In [ ]:
process_mrio(
    "exiobase_3102_ixi",
    agg_sec=True,
    agg_version="oecd_d",
    refresh=False,
)

In [ ]:
process_mrio(
    "oecd_v2025",
    agg_reg=True,
    agg_version="NL",
    refresh=False,
)

#### Computing deterministic aSoCC target

In [ ]:
# OECD ICIO sector D aSoCC target
deterministic_asocc(
    project_name=project_name_,
    source="oecd_v2025",
    agg_reg=True,
    agg_version="NL",
    years=study_period_,
    fu_code=fu_code_,
    s_p=["D"] if fu_code_ in FU_FOR_S_P else None,
    r_c=r_c_ if fu_code_ in FU_FOR_R_C else None,
    r_f=r_f_ if fu_code_ in FU_FOR_R_F else None,
    r_p=r_p_ if fu_code_ in FU_FOR_R_P else None,
    figures=False,
    refresh=False,
)

# EXIOBASE grouped to OECD ICIO sector D
deterministic_asocc(
    project_name=project_name_,
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="oecd_d",
    years=study_period_,
    fu_code=fu_code_,
    s_p=["D"] if fu_code_ in FU_FOR_S_P else None,
    r_c=r_c_ if fu_code_ in FU_FOR_R_C else None,
    r_f=r_f_ if fu_code_ in FU_FOR_R_F else None,
    r_p=r_p_ if fu_code_ in FU_FOR_R_P else None,
    figures=False,
    refresh=False,
)

# EXIOBASE electricity reference
deterministic_asocc(
    project_name=project_name_,
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="ELECTRICITY",
    years=study_period_,
    fu_code=fu_code_,
    s_p=s_p_ if fu_code_ in FU_FOR_S_P else None,
    r_c=r_c_ if fu_code_ in FU_FOR_R_C else None,
    r_f=r_f_ if fu_code_ in FU_FOR_R_F else None,
    r_p=r_p_ if fu_code_ in FU_FOR_R_P else None,
    lcia_method=lcia_method_,
    figures=False,
    refresh=False,
)

#### Disaggregate OECD ICIO sector D to EXIOBASE electricity


In [ ]:
if "L2" in fu_code_:
    disaggregate_asocc(
        disaggregation_config={
            "target_agg_run": {
                "source": "oecd_v2025",
                "agg_reg": True,
                "agg_version": "NL",
                "s_p": ["D"],
            },
            "ref_agg_run": {
                "source": "exiobase_3102_ixi",
                "agg_sec": True,
                "agg_version": "oecd_d",
                "s_p": ["D"],
            },
            "ref_disagg_run": {
                "source": "exiobase_3102_ixi",
                "agg_sec": True,
                "agg_version": "ELECTRICITY",
                "s_p": s_p_,
            },
            "disaggregation_specs": [
                {"agg_sector_label": "D", "disagg_sector_label": "ELECTRICITY"}
            ],
            "new_disagg_version_name": "oecd_electricity",
        },
        base_asocc_args={
            "project_name": project_name_,
            "years": study_period_,
            "fu_code": fu_code_,
            "r_c": r_c_ if fu_code_ in FU_FOR_R_C else None,
            "r_f": r_f_ if fu_code_ in FU_FOR_R_F else None,
            "r_p": r_p_ if fu_code_ in FU_FOR_R_P else None,
        },
        figures=False,
        refresh=False,
    )

# Run ASR uncertainty 

`pb_lcia` with static carrying capacities, `gwp100_lcia` with dynamic AR6 carrying capacities, then `gwp100_lcia` with static carrying capacities.
Sobol is deactivated for all cases, while Monte Carlo convergence settings use pyaesa defaults.

### static CC

In [ ]:
lcia_method_sub_ = lcia_method_ if fu_code_ not in ["L2.a.c"] else "gwp100_lcia"

In [ ]:
print(f"Running {lcia_method_sub_} | static CC")

uncertainty_asr(
    project_name=project_name_,
    source=source_,
    years=study_period_,
    fu_code=fu_code_,
    agg_reg=agg_reg_,
    agg_sec=agg_sec_,
    agg_version=agg_version_,
    s_p=s_p_ if fu_code_ in FU_FOR_S_P else None,
    r_c=r_c_ if fu_code_ in FU_FOR_R_C else None,
    r_f=r_f_ if fu_code_ in FU_FOR_R_F else None,
    r_p=r_p_ if fu_code_ in FU_FOR_R_P else None,
    lcia_method=lcia_method_sub_,
    lca_args={"external_lca": {"active": False}, "io_lca": {"active": True}},
    uncertainty_config={
        "mc_parameters": {
            "fixed": {"active": not mc_convergence_, "n_runs": mc_n_runs_},
            "convergence": {"active": mc_convergence_, "max_runs": mc_max_runs_},
        },
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": sector_cov_mapping_,
            },
            "inter_mrio_uncertainty": {
                "active": True
                if "L2" in fu_code_
                else False,  # TODO: ERROR "no objects to concatenate" for L1 allocation level with inter-mrio uncertainty included
                "alternate_source": "oecd_electricity"
                if "L2" in fu_code_
                else "oecd_v2025_NL"
                if "L1" in fu_code_
                else None,
            },
            "reference_year_uncertainty": {"active": True},
            "inter_method_uncertainty": {"active": True, "mode": "equal_weight"},
        },
        "io_lca_uncertainty_sources": {
            "lcia_uncertainty": {
                "sector_cov_mapping": sector_cov_mapping_,
            },
        },
    },
    sobol_parameters={
        "active": sobol_active_,
        "fixed": {"active": True, "n_base_samples": 128},
        "convergence": {
            "active": False,
            "rtol": 0.05,
        },
    },
    output_format="csv_compact",  # csv_compact
    # figure_format={"format": "svg"},
    figure_options={
        "per_method": False,
        "multi_method": True,
        "inter_method": True,
        "polar": {
            "active": False if fu_code_ in ["L2.a.c"] else True,
            "polar_years": list(study_period_),
            "polar_style": "violin",
        },
    },  # TODO: ERROR while running the pb_lcia violin for fu=L2.a.c: polar plots are skipped for L2.a.c for now.
    subfigures=subfigures_,
    refresh=True,
)

### dynamic CC: `gwp100_lcia` dynamic AR6 carrying capacities


In [ ]:
if run_dynamic_ar6_b:
    print("Running gwp100_lcia | dynamic AR6 CC")
    uncertainty_asr(
        project_name=project_name_ + "_dynamic",
        source=source_,
        years=dynamic_study_period_,
        fu_code=fu_code_,
        agg_reg=agg_reg_,
        agg_sec=agg_sec_,
        agg_version=agg_version_,
        s_p=s_p_ if fu_code_ in FU_FOR_S_P else None,
        r_c=r_c_ if fu_code_ in FU_FOR_R_C else None,
        r_f=r_f_ if fu_code_ in FU_FOR_R_F else None,
        r_p=r_p_ if fu_code_ in FU_FOR_R_P else None,
        lcia_method="gwp100_lcia",
        base_asocc_args={"ssp_scenario": ["SSP2"]},
        base_cc_args={
            "static": {"active": False},
            "dynamic_ar6": {"active": True, "ssp_scenario": ["SSP2"]},
        },
        lca_args={"external_lca": {"active": False}, "io_lca": {"active": True}},
        uncertainty_config={
            "mc_parameters": {"convergence": {"max_runs": mc_max_runs_}},
            "asocc_uncertainty_sources": {
                "lcia_uncertainty": {
                    "active": True,
                    "sector_cov_mapping": sector_cov_mapping_,
                },
                "inter_mrio_uncertainty": {
                    # This could be changed to True to include inter-mrio uncertainty,
                    # but this then requires running the section 'Preparing disaggregation...'
                    # with years=dynamic_study_period_
                    "active": False,
                    "alternate_source": "oecd_electricity",
                },
                "reference_year_uncertainty": {"active": True},
                "inter_method_uncertainty": {"active": True, "mode": "equal_weight"},
            },
            "io_lca_uncertainty_sources": {
                "lcia_uncertainty": {
                    "sector_cov_mapping": sector_cov_mapping_,
                },
            },
            "ar6_cc_uncertainty_sources": {
                "dynamic_ar6_cc_uncertainty": {"category_uncertainty": True},
            },
        },
        sobol_parameters={"active": True},
        output_format="parquet",
        figure_options={
            "per_method": False,
            "multi_method": True,
            "inter_method": True,
            "polar": {"active": True, "polar_years": None, "polar_style": "violin"},
        },
        figure_format={"format": "png", "dpi": 1000},
    )


# Generate GIF file with polar plots

If you want to save the GIF in another format than .gif, make sure to install from the command line:\
``pip install imageio-ffmpeg`` or ``pip install imageio[ffmpeg]`` \
``pip install av`` \

Note that this functionnality will soon be available directly in ``pyaesa``.

In [ ]:
def get_polar_files(workspace=WORKSPACE_TOP, project_name=project_name_):

    asr_dir = f"{workspace.replace('\\', '/')}/{project_name}/C_asr"
    asr_files = glob.glob("**/mc_*", root_dir=asr_dir, recursive=True)
    relevant_asr_files = [
        f"{asr_dir}/{f.replace('\\', '/')}"
        for f in asr_files
        if ("pb_lcia" in f) and ("io_lca" in f)
    ]
    relevant_asr_files.sort(key=os.path.getmtime)
    polars_files_p = glob.glob("**/pol_vio_*", root_dir=relevant_asr_files[-1], recursive=True)
    polar_files = [
        f"{relevant_asr_files[-1]}/{f.replace('\\', '/')}"
        for f in polars_files_p
        if (".png" in f) or (".svg" in f)
    ]

    return polar_files


def create_gif(filenames, save_path):

    images = []
    for filename in filenames:
        images.append(iio.imread(filename))

    if ".gif" not in save_path:
        iio.imwrite(save_path, images, fps=5)
    else:
        iio.imwrite(save_path, images, duration=200, loop=0)

    return None

In [ ]:
if generate_gif_b and "pb_lcia" in lcia_method_sub_:
    print(f"Generating a GIF for fu={fu_code_} based on polar plots (pb_lcia)")

    polar_files = get_polar_files(workspace=WORKSPACE_TOP, project_name=project_name_)
    create_gif(
        filenames=polar_files, save_path=f"{polar_files[0].split(str(study_period_[0]))[0]}GIF.mp4"
    )

# Statistics

In [ ]:
timestamp_stop = datetime.now()
total_execution_time = timestamp_stop - timestamp_start

print(f"Completion for fu={fu_code_} at: {timestamp_stop}")
print(f"Total execution time: {total_execution_time}")